# Airflow 3: What Changed

**Airflow 3.0** (April 2025) was the largest release since 2.0. Many tutorials and Stack Overflow answers still show Airflow 2 code. This lesson explains the main changes and demonstrates two of them on the running Airflow 3 instance: **assets** (data-aware scheduling) and **DAG versioning**.

**After this lesson you can:**

1. List the main architectural and API changes of Airflow 3.
2. Schedule DAGs on **assets** produced by other DAGs.
3. Inspect DAG versions through the REST API v2.
4. Migrate Airflow 2 code with a checklist.

In [1]:
import sys
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
datasets.ensure_service("orchestration", "airflow", timeout=900)      # Airflow 3 standalone in Docker (first start: a few minutes)
API = "http://localhost:8080/api/v2"
DAGS = datasets.ROOT / "data" / "cache" / "airflow" / "dags"          # mounted into the container as /opt/airflow/dags
OUT = datasets.ROOT / "data" / "cache" / "airflow" / "output"         # mounted as /opt/airflow/output
DATA_IN_CONTAINER = "/opt/airflow/course_data"                        # the course data cache, read-only

def deploy(dag_id: str, source: str, timeout: int = 120) -> None:
    """Write a DAG file and wait until Airflow has parsed it; then unpause it."""
    (DAGS / f"{dag_id}.py").write_text(textwrap.dedent(source).strip() + "\n", encoding="utf-8")
    deadline = time.time() + timeout
    while time.time() < deadline:
        r = requests.get(f"{API}/dags/{dag_id}", timeout=10)
        if r.status_code == 200:
            requests.patch(f"{API}/dags/{dag_id}", json={"is_paused": False}, timeout=10)
            return
        time.sleep(3)
    errors = requests.get(f"{API}/importErrors", timeout=10).json()
    raise RuntimeError(f"DAG {dag_id} was not parsed in time; import errors: {errors}")

def trigger(dag_id: str, conf: dict | None = None) -> str:
    r = requests.post(f"{API}/dags/{dag_id}/dagRuns", json={"logical_date": None, "conf": conf or {}}, timeout=10)
    r.raise_for_status()
    return r.json()["dag_run_id"]

def wait(dag_id: str, run_id: str, timeout: int = 300) -> str:
    deadline = time.time() + timeout
    while time.time() < deadline:
        state = requests.get(f"{API}/dags/{dag_id}/dagRuns/{run_id}", timeout=10).json()["state"]
        if state in ("success", "failed"):
            return state
        time.sleep(3)
    return "timeout"

def tasks(dag_id: str, run_id: str) -> pd.DataFrame:
    ti = requests.get(f"{API}/dags/{dag_id}/dagRuns/{run_id}/taskInstances", params={"limit": 100}, timeout=10).json()["task_instances"]
    df = pd.DataFrame(ti)[["task_id", "map_index", "state", "try_number", "start_date", "end_date", "duration"]]
    for c in ["start_date", "end_date"]:
        df[c] = pd.to_datetime(df[c])
    return df.sort_values("start_date").reset_index(drop=True)

def xcom(dag_id: str, run_id: str, task_id: str, map_index: int = -1):
    r = requests.get(f"{API}/dags/{dag_id}/dagRuns/{run_id}/taskInstances/{task_id}/xcomEntries/return_value",
                     params={"map_index": map_index, "deserialize": True}, timeout=10)
    return r.json().get("value")

def gantt(df: pd.DataFrame, title: str) -> None:
    t0 = df["start_date"].min()
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(df) + 1))
    colors = {"success": "#59A14F", "failed": "#E15759", "skipped": "#BAB0AC", "upstream_failed": "#F28E2B"}
    labels = [f"{t}[{m}]" if m >= 0 else t for t, m in zip(df["task_id"], df["map_index"])]
    for i, r in df.iterrows():
        if pd.notna(r["start_date"]):
            ax.barh(i, (r["end_date"] - r["start_date"]).total_seconds(), left=(r["start_date"] - t0).total_seconds(),
                    color=colors.get(r["state"], "#4E79A7"))
    ax.set(yticks=range(len(df)), yticklabels=labels, xlabel="Seconds since the first task started", title=title)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

health = requests.get(f"{API}/monitor/health", timeout=10).json()
print("Airflow components:", {k: v["status"] for k, v in health.items()})
print("Airflow version:", requests.get(f"{API}/version", timeout=10).json()["version"])

Airflow components: {'metadatabase': 'healthy', 'scheduler': 'healthy', 'triggerer': 'healthy', 'dag_processor': 'healthy'}
Airflow version: 3.1.0


## 1. Main Changes

| Area | Airflow 2 | Airflow 3 |
| :--- | :--- | :--- |
| DAG authoring imports | `airflow.decorators`, `airflow.models` | **Task SDK**: `from airflow.sdk import dag, task, Asset` |
| task execution | workers talk to the metadata database directly | tasks talk to an **API server** (Task Execution Interface); remote/edge executors possible |
| data-aware scheduling | "Datasets" (2.4+) | **Assets** with names, URIs, groups, asset watchers |
| DAG history | the latest code only | **DAG versioning**: runs keep the version they started with |
| backfills | CLI process outside the scheduler | **scheduler-managed** backfills via API/UI |
| logical date | required for every run | can be `None` (event-driven, manual runs) |
| REST API | `/api/v1` | `/api/v2` (FastAPI) |
| UI | Flask-AppBuilder | new React UI |
| removed | SubDAGs, SLAs, `execution_date` in templates, `schedule_interval` | use task groups, deadline alerts, `logical_date`, `schedule` |
| default `catchup` | True | **False** |

## 2. Assets: Data-Aware Scheduling

A producer task declares `outlets=[asset]`; a consumer DAG uses `schedule=[asset]` and runs **whenever the asset is updated**, regardless of clock time. This links pipelines by data instead of by guessing times ("run the report at 07:00 because the load usually ends at 06:30").

In [2]:
deploy("asset_producer", f"""
from datetime import datetime
from airflow.sdk import Asset, dag, task

DAILY_COUNTS = Asset(name="taxi_daily_counts", uri="file:///opt/airflow/output/asset_daily_counts.parquet")

@dag(schedule=None, start_date=datetime(2024, 1, 1), catchup=False, tags=["course", "m11", "assets"])
def asset_producer():

    @task(outlets=[DAILY_COUNTS])
    def build_daily_counts():
        import duckdb
        duckdb.execute(\"\"\"COPY (SELECT CAST(tpep_pickup_datetime AS DATE) AS day, count(*) AS trips
                          FROM read_parquet('{DATA_IN_CONTAINER}/yellow_tripdata_2024-02.parquet')
                          WHERE tpep_pickup_datetime >= DATE '2024-02-01' GROUP BY 1)
                          TO '/opt/airflow/output/asset_daily_counts.parquet' (FORMAT parquet)\"\"\")

    build_daily_counts()

asset_producer()
""")
deploy("asset_consumer", """
from datetime import datetime
from airflow.sdk import Asset, dag, task

DAILY_COUNTS = Asset(name="taxi_daily_counts", uri="file:///opt/airflow/output/asset_daily_counts.parquet")

@dag(schedule=[DAILY_COUNTS], start_date=datetime(2024, 1, 1), catchup=False, tags=["course", "m11", "assets"])
def asset_consumer():

    @task
    def report(**context):
        import duckdb
        events = context["triggering_asset_events"]
        day, trips = duckdb.execute("SELECT day, trips FROM '/opt/airflow/output/asset_daily_counts.parquet' "
                                    "ORDER BY trips DESC LIMIT 1").fetchone()
        return {"triggered_by": [str(k) for k in events.keys()], "busiest_day": str(day), "trips": int(trips)}

    report()

asset_consumer()
""")
before = len(requests.get(f"{API}/dags/asset_consumer/dagRuns", timeout=10).json()["dag_runs"])
rid = trigger("asset_producer")
print("producer:", wait("asset_producer", rid))
deadline = time.time() + 120
while time.time() < deadline:
    runs = requests.get(f"{API}/dags/asset_consumer/dagRuns", timeout=10).json()["dag_runs"]
    if len(runs) > before:
        break
    time.sleep(3)
crun = sorted(runs, key=lambda r: r["queued_at"])[-1]
print("consumer run created automatically:", crun["run_type"], "->", wait("asset_consumer", crun["dag_run_id"]))
print("consumer result:", xcom("asset_consumer", crun["dag_run_id"], "report"))
assets = requests.get(f"{API}/assets", timeout=10).json()["assets"]
print("registered assets:", [(a["name"], a["uri"]) for a in assets])

producer: success


consumer run created automatically: asset_triggered -> success
consumer result: {'trips': 129890, 'busiest_day': '2024-02-29', 'triggered_by': ["Asset(name='taxi_daily_counts', uri='file:///opt/airflow/output/asset_daily_counts.parquet', group='asset', extra={}, watchers=[])"]}
registered assets: [('taxi_daily_counts', 'file:///opt/airflow/output/asset_daily_counts.parquet')]


The consumer run has the run type `asset_triggered`: nobody scheduled it by time; the producer's successful task emitted an **asset event**.

## 3. DAG Versioning

When a DAG file changes, Airflow 3 stores a new **DAG version**. Runs that already started keep the version they started with, and the UI can show the graph and code of any past run. The cell changes the producer DAG (adds a tag) and lists its versions.

In [3]:
def n_versions() -> int:
    return len(requests.get(f"{API}/dags/asset_producer/dagVersions", timeout=10).json()["dag_versions"])

before_versions = n_versions()
src = (DAGS / "asset_producer.py").read_text()
(DAGS / "asset_producer.py").write_text(src.replace('tags=["course", "m11", "assets"]', 'tags=["course", "m11", "assets", "v2"]'))
deadline = time.time() + 120
while n_versions() == before_versions and time.time() < deadline:   # wait until the DAG processor stored the new version
    time.sleep(3)
rid2 = trigger("asset_producer")
wait("asset_producer", rid2)
versions = requests.get(f"{API}/dags/asset_producer/dagVersions", timeout=10).json()["dag_versions"]
print(pd.DataFrame(versions)[["version_number", "created_at"]].to_string(index=False))
runs = requests.get(f"{API}/dags/asset_producer/dagRuns", timeout=10).json()["dag_runs"]
print("\nruns and the DAG version they used:")
for r in sorted(runs, key=lambda r: r["queued_at"]):
    print(f"  {r['dag_run_id'][:32]}  versions: {[v['version_number'] for v in r.get('dag_versions', [])]}")

 version_number                  created_at
              1 2026-11-16T20:21:37.214828Z
              2 2026-11-16T20:26:21.739544Z
              3 2026-11-16T20:27:04.260500Z
              4 2026-11-17T11:48:42.104055Z

runs and the DAG version they used:
  manual__2026-11-16T20:26:21.3352  versions: [2]
  manual__2026-11-16T20:27:00.2148  versions: [2]
  manual__2026-11-17T11:47:52.7546  versions: [3]
  manual__2026-11-17T11:48:42.7673  versions: [4]


## 4. Migration Checklist (Airflow 2 → 3)

| Step | How |
| :--- | :--- |
| update imports to the Task SDK | `from airflow.sdk import dag, task, Asset` |
| rename deprecated parameters | `schedule_interval` → `schedule`; `execution_date` → `logical_date` |
| replace removed features | SubDAGs → task groups; SLAs → deadline alerts; `Dataset` → `Asset` |
| no direct metadata-DB access in tasks | use the API / Task SDK instead of `settings.Session` |
| check `catchup` assumptions | default is now `False` |
| lint DAGs automatically | Ruff rules `AIR3xx` flag Airflow 3 incompatibilities |
| test on a staging Airflow 3 | `dag.test()` and import checks in CI |

*Insight: Airflow 3 moves tasks behind an API, schedules pipelines by data (assets) as well as by time, versions every DAG change, and brings backfills into the scheduler. New DAGs should use the Task SDK and assets; old DAGs need a deliberate migration, checked by linters and a staging deployment.*

In [4]:
for dag_id in ["taxi_daily_summary", "taxi_by_borough", "branching_and_retries", "wait_for_delivery", "daily_partition",
               "asset_producer", "asset_consumer"]:
    requests.patch(f"{API}/dags/{dag_id}", json={"is_paused": True}, timeout=10)
print("course DAGs paused; stop Airflow when you are done: docker compose -f docker/compose.yml --profile orchestration stop")

course DAGs paused; stop Airflow when you are done: docker compose -f docker/compose.yml --profile orchestration stop


## Real-World Use Case or Analogy

Asset scheduling is like **a kitchen that starts the dessert when the cake comes out of the oven**, not at a fixed time: if the cake is late, dessert waits; if it is early, dessert starts early. DAG versioning is like **keeping every edition of a recipe book**, so you always know which recipe produced yesterday's dish.

Companies upgrading from Airflow 2 typically run both versions side by side during migration; managed services (Astronomer, MWAA, Cloud Composer) offer Airflow 3 environments.